In [ ]:
import os
DREAMS_DATA = os.environ.get("DREAMS_DATA", "/path/to/DREAMS")
TORREYLABTOOLS = os.environ.get("TORREYLABTOOLS", "/path/to/torreylabtools")

import numpy as np
import matplotlib.pyplot as plt
import h5py, os
from matplotlib.colors import LogNorm
from scipy.spatial import KDTree
from scipy.optimize import curve_fit
from photutils.isophote import Ellipse, EllipseGeometry
from scipy.stats import binned_statistic

In [ ]:
def load_group_data(path, keys):
    """
    Read Group Data from the DREAMS simulations
    
    Inputs
      path - the absolute or relative path to the hdf5 file you want to read from
      keys - the data that you want to read from the simulation 
             see https://www.tng-project.org/data/docs/specifications/ for a list of available data)
      
    Returns
      cat - a dictionary that contains all of the group and subhalo information for the specified keys
    """
    
    cat = dict()
    with h5py.File(path) as ofile:
        for key in keys:
            if 'Group' in key:
                cat[key] = np.array(ofile[f'Group/{key}'])
            if 'Subhalo' in key:
                cat[key] = np.array(ofile[f'Subhalo/{key}'])
    return cat

def get_MW_idx(cat):
    """
    Selects the corrent MW-mass galaxy from each simulation given the group catalog.
    This function only works for z~0
    It selects the least contaminated halo with a mass within current uncertainties of the MW's mass
    
    Inputs 
     - cat - a dictionary containing the 'GroupMassType' field from the FOF catalogs
     
    Returns
     - mw_idx - the index into the group catalog for the target MW-mass galaxy
    """
    masses = cat['GroupMassType'] * 1e10 / h
    
    tot_masses = np.sum(masses,axis=1)
    mcut = (tot_masses > 5e11) & (tot_masses < 2.5e12)
    
    contamination = masses[:,2] / tot_masses
    idx = np.argmin(contamination[mcut])
    
    mw_idx = np.arange(len(masses))[mcut][idx]
    
    return mw_idx

def get_sats(cat, mw_idx, rlim=300, pos_cut=False):
    """
    This function takes group catalog information and the index for the target MW-mass galaxy 
    and produces a new dictionary that contains the same information, but only for the satellites around that system.
    This function also applies a distance and mass cut to the satellites that it returns.
    
    Inputs
     - cat - a dictionary containing the information from the FOF catalogs
             the required fields are: GroupPos, SubhaloPos, SubhaloMassType, SubhaloGrNr
             see https://www.tng-project.org/data/docs/specifications/ for a description of these fields
     - mw_idx - the index of the MW-mass galaxy into the FOF catalog
     
    Returns
     - sub_cat - a new dictionary containing FOF and Subfind data for the satellites around the target MW galaxy
                 the new dictionary will contain all fields containted within cat
                 all group fields will only contain information for the target MW galaxy
    """
    if pos_cut:
        pos = cat['GroupPos'][mw_idx]
        r = np.sqrt(np.sum(np.square(cat['SubhaloPos'] - pos),axis=1))
        mwcut = r < rlim * h
        rcut = mwcut
    else:
        mwcut = cat['SubhaloGrNr'] == mw_idx
        r = np.sqrt(np.sum(np.square(cat['GroupPos'][mw_idx] - cat['SubhaloPos']), axis=1))
        rcut = r < rlim * h

    mcut = cat['SubhaloMass'] * 1e10 / h>= mass_thresh
    bcut = cat['SubhaloMassType'][:,4] * 1e10 / h>= stellar_mass_thresh    
    
    cut = mwcut & mcut & rcut & bcut
    
    sub_cat = dict()
    for key in cat:
        if 'Group' in key:
            sub_cat[key] = cat[key][mw_idx]
        if 'Subhalo' in key:
            sub_cat[key] = cat[key][cut]
        
    return sub_cat

def get_params(file):
    """
    This function reads in the simulation parameters for all DREAMS sims and converts them into physical units
    
    Inputs
     - file - absolute or relative path to the file containing all simulation parameters
     
    Returns
     - params - an Nx4 ndarray with the following parameters
                index 0 - WDM particle mass [keV]
                index 1 - SN1 (ew) wind energy
                index 2 - SN2 (kw) wind velocity
                index 3 - AGN1 (BHFF) black hole feedback factor
    """
    sample = np.loadtxt(file)
    
    fiducial = np.array([1,3.6,7.4,.1]) #fiducial TNG values
    params = sample * fiducial
    params[:,0] = 1/params[:,0]
    
    return params

def get_params_cdm(file):
    """
    This function reads in the simulation parameters for all DREAMS sims and converts them into physical units
    
    Inputs
     - file - absolute or relative path to the file containing all simulation parameters
     
    Returns
     - params - an Nx4 ndarray with the following parameters
                index 0 - WDM particle mass [keV]
                index 1 - SN1 (ew) wind energy
                index 2 - SN2 (kw) wind velocity
                index 3 - AGN1 (BHFF) black hole feedback factor
    """
    sample = np.loadtxt(file)
    
    fiducial = np.array([1,1,3.6,7.4,.1]) #fiducial TNG values
    params = sample * fiducial
    
    return params

def norm_params(params):
    """
    This function normalizes the four simulation parameters (WDN, SN1, SN2, AGN).
    
    Inputs
     - params - an Nx4 array of simulation parameters
    
    Results
     - nparams - same as the input but now normalized and linearly sampled between 0 and 1
    """
    nparams = params / np.array([1, 3.6, 7.4, .1])
    nparams[:,0] = 1/nparams[:,0]
    
    nparams[:,1:] = np.log10(nparams[:,1:])

    minimum = np.array([1/30, 0.25, 0.5, 0.25])
    maximum = np.array([1/1.8, 4.0, 2.0, 4.0])

    nparams = (nparams - minimum)/(maximum - minimum)

    nparams[:,1:] = np.power(10, nparams[:,1:])

    return nparams

def norm_params_cdm(params):
    """
    This function normalizes the four simulation parameters (WDN, SN1, SN2, AGN).
    
    Inputs
     - params - an Nx4 array of simulation parameters
    
    Results
     - nparams - same as the input but now normalized and linearly sampled between 0 and 1
    """
    nparams = params / np.array([1, 1, 3.6, 7.4, .1])
    
    nparams[:,2:] = np.log10(nparams[:,2:])

    minimum = np.array([.274, .780, 0.25, 0.5, 0.25])
    maximum = np.array([.354, .888, 4.0, 2.0, 4.0])

    nparams = (nparams - minimum)/(maximum - minimum)

    nparams[:,2:] = np.power(10, nparams[:,2:])

    return nparams

def normalize_data(data):
    """
    This funciton normalizes the input data for the emulator
    
    Inputs
     - data - an array of values measured from the simulations
     
    Returns
     - out - the normalized array such that the mean = 0 and std = 1
    """
    out = np.log10(data + 1)
    mean = np.mean(out, axis=0)
    std = np.std(out, axis=0)
    out = (out - mean) / std
    return out

def load_particle_data(path, keys, part_types):
    """
    Read particle data from the DREAMS simulations
    
    Inputs
      path - the absolute or relative path to the hdf5 file you want to read from
      keys - the data that you want to read from the simulation 
             see https://www.tng-project.org/data/docs/specifications/ for a list of available data)
      part_types - which particle types to load.
                   0 - gas
                   1 - high res dark matter
                   2 - low res dark matter
                   3 - tracers (not used in DREAMS)
                   4 - stars
                   5 - black holes
      
    Returns
      cat - a dictionary that contains all of the particle information for the specified keys and particle types
    """
    cat = dict()
    with h5py.File(path) as ofile:
        
        if type(part_types) == type(0):
            part_types = [part_types]
        
        for pt in part_types:
            for key in keys:
                if pt == 1 and key == 'Masses':
                    cat[f'PartType{pt}/{key}'] = np.ones(ofile['PartType1/ParticleIDs'].shape)*ofile['Header'].attrs['MassTable'][1]
                else:
                    if f'PartType{pt}/{key}' in ofile:
                        cat[f'PartType{pt}/{key}'] = np.array(ofile[f'PartType{pt}/{key}'])
    return cat

def get_galaxy_data(part_cat, group_cat, fof_idx=-1, sub_idx=-1):
    """
    Given particle and group catalogs, return a new catalog that only contains data for a specified galaxy.
    If fof_idx is given but sub_idx is not, data for the FOF group and all satellites are returned
    If fof_idx is given and sub_idx is given, data for just the specified subhalo of that group
      e.g. fof_idx=3 sub_idx=5 will provide data for the fifth subhalo of group three
    If only sub_idx is supplied, the data for that subfind galaxy is returned, can be in any FOF group
    
    Inputs
      part_cat  - a dictionary containing particle data make from load_particle_data
      group_cat - a dictionary containing group data make from load_particle_data
                  must contain these fields: GroupLenType, GroupFirstSub, GroupNsubs, SubhaloLenType, SubhaloGrNr
      fof_idx   - the FOF group that you want data for
      sub_idx   - the Subfind galaxy that you want data for
      
    Returns
      new_part_cat  - a new dictionary with keys from part_cat but data only for the specified galaxy
      new_group_cat - a new dictionary with keys from group_cat but data only for the specified galaxy
    """
    
    if fof_idx < 0 and sub_idx < 0:
        return part_cat, group_cat
    
    if fof_idx < 0 and sub_idx >= 0:
        fof_idx = group_cat['SubhaloGrNr'][sub_idx]

    if sub_idx >= 0:
        if sub_idx < group_cat['GroupFirstSub'][fof_idx]:
            sub_idx += group_cat['GroupFirstSub'][fof_idx]
    
    offsets = np.sum(group_cat['GroupLenType'][:fof_idx],axis=0)
    
    if sub_idx >= 1:
        start_sub = group_cat['GroupFirstSub'][fof_idx]
        offsets += np.sum(group_cat['SubhaloLenType'][start_sub:sub_idx], axis=0)
    
    if sub_idx < 0:
        num_parts = group_cat['GroupLenType'][fof_idx]
        nsubs = group_cat['GroupNsubs'][fof_idx]
        sub_start = group_cat['GroupFirstSub'][fof_idx]
    else:
        num_parts = group_cat['SubhaloLenType'][sub_idx]
        nsubs = 1
        sub_start = sub_idx
    
    new_part_cat = dict()
    for key in part_cat:
        pt = int(key.split("/")[0][-1])
        new_part_cat[key] = part_cat[key][offsets[pt]:offsets[pt]+num_parts[pt]]
    
    new_group_cat = dict()
    for key in group_cat:
        if 'Group' in key:
            new_group_cat[key] = group_cat[key][fof_idx]
        else:
            if nsubs == 1:
                new_group_cat[key] = group_cat[key][sub_start]
            else:
                new_group_cat[key] = group_cat[key][sub_start:sub_start+nsubs]
    
    return new_part_cat, new_group_cat

In [ ]:
snap_path = DREAMS_DATA + '/Sims/CDM/MW_zooms/SB5/'
group_path = DREAMS_DATA + '/FOF_Subfind/CDM/MW_zooms/SB5/'
param_path = DREAMS_DATA + '/Parameters/CDM/MW_zooms/CDM_TNG_MW_SB5.txt'

In [ ]:
keys = ['GroupMassType', 'GroupLenType', 'GroupPos', 'SubhaloGrNr', 'SubhaloPos', 'SubhaloMassType', 'SubhaloLenType', 'GroupFirstSub', 'GroupNsubs']
keys += ['SubhaloMass', 'SubhaloSFR', 'SubhaloPos', 'SubhaloVel', 'GroupVel', 'SubhaloHalfmassRadType', 'GroupMass', 'SubhaloStellarPhotometrics', 'SubhaloStarMetalFractions', 'SubhaloGasMetalFractions', 'SubhaloVelDisp', 'SubhaloGasMetallicity','Group_M_Crit200']
part_keys = ['Coordinates', 'Masses', 'Velocities', 'GFM_StellarPhotometrics', 'GFM_StellarFormationTime']

In [ ]:
name = "NeHOD_master"              #name that files will be saved as. Should be different for every unique project/application
boxes = range(1024)        #which simulations you include, there are 1024 simulations
snap = 90                  #which redshift you use, right now the script only works for snap >~ 60
h = 0.6909                 #reduced hubble factor, don't change this
mass_thresh = 1e7          #only include satellites above this halo mass
stellar_mass_thresh = 0 #1e6  #only include satellites above this stellar mass
boxsize = 100000 / h 
G = 4.3e-6                 #kpc/Msun * (km/s)^2

In [ ]:
params = get_params_cdm(param_path)
# params = get_params(param_path)
params = np.array(params)

In [ ]:
def calc_sfh(pcat, cat, mw_idx, sat_idx):

    a10 = []
    a50 = []
    a90 = []
    a100 = []
    for idx in sat_idx:

        sub_idx = idx - cat['GroupFirstSub'][mw_idx]
        sat_pcat, sat_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx, sub_idx=sub_idx)

        ages = sat_pcat['PartType4/GFM_StellarFormationTime']
        masses = sat_pcat['PartType4/Masses']

        if np.sum(ages > 0) == 0: #all stars are wind particles
            a10.append(0)
            a50.append(0)
            a90.append(0)
            a100.append(0)
            continue
        
        cut = ages > 0 #wind particles
        ages = ages[cut]
        masses = masses[cut]

        tot_mass = np.sum(masses)
    
        idx = np.argsort(ages)
        cs_mass = np.cumsum(masses[idx]/tot_mass)
    
        a10.append(ages[idx][np.argmin(np.abs(cs_mass-0.1))])
        a50.append(ages[idx][np.argmin(np.abs(cs_mass-0.5))])
        a90.append(ages[idx][np.argmin(np.abs(cs_mass-0.9))])
        a100.append(ages[idx][np.argmin(np.abs(cs_mass-1))])
            
    return a10, a50, a90, a100

def get_sat_idx(cat, mw_idx):

    mwcut = cat['SubhaloGrNr'] == mw_idx
    r = np.sqrt(np.sum(np.square(cat['GroupPos'][mw_idx] - cat['SubhaloPos']), axis=1))
    rcut = r < 400 * h

    mcut = cat['SubhaloMass'] * 1e10 / h>= mass_thresh
    bcut = cat['SubhaloMassType'][:,4] * 1e10 / h>= stellar_mass_thresh    
    
    cut = mwcut & mcut & rcut & bcut
    
    return np.arange(len(cut))[cut][1:]

def calc_density(matter_coords, matter_mass, sample_coords, DesNgb):
    """
    This function takes particle data and calculates the density around a given set of points.
    The points are assumed to be for one radius and are averaged together before returned.
    This function will work for any particle type given to it.
    
    Inputs
      matter_coords - the coordinates for the simulation particles used to calculate the density
      matter_mass   - the mass of the simulation particles, must be the same shape as matter_coords
      sample_coords - the coordinates where the density is calculated
      DesNgb        - the number of simulation particles used to calculate the density (32 is standard)
      
    Returns
      density - the average density for the supplied sample coordinates
    """
    tree = KDTree(matter_coords)
    distance, idx = tree.query(sample_coords, DesNgb)
    hsml = distance[:,-1]
    mass_enclosed = np.sum(matter_mass[idx], axis=1)
    density =  mass_enclosed / (4 / 3 * np.pi * np.power(hsml,3))
    
    density = np.average(density)
    return density

def fibonacci_sphere(samples, r):
    """
    This function returns a set of points equally spaced on the surface of a sphere.
    This function was adapted from https://stackoverflow.com/questions/9600801/evenly-distributing-n-points-on-a-sphere
    
    Inputs
      samples - the number of points on the sphere
      r       - the radius of the sphere that is sampled
      
    Returns
      points  - the coordinates of the points of the sphere with shape (samples,3)
    """
    points = []
    phi = np.pi * (np.sqrt(5.) - 1.)  # golden angle in radians

    for i in range(samples):
        y = 1 - (i / (samples - 1)) * 2  
        radius = np.sqrt(1 - y * y) 

        theta = phi * i  # golden angle increment

        x = np.cos(theta) * radius
        z = np.sin(theta) * radius

        points.append((x, y, z))

    points = np.array(points) * r
        
    return points

def calc_dm(pcat, cat, mw_idx, sat_idx):

    dens = []
    slope = []
    for idx in sat_idx:

        sub_idx = idx - cat['GroupFirstSub'][mw_idx]
        sat_pcat, sat_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx, sub_idx=sub_idx)

        coords = (sat_pcat['PartType1/Coordinates'] - sat_cat['SubhaloPos']) / h
        # print(mw_idx, idx, sub_idx, np.sum(np.average(coords,axis=0)))
        masses = sat_pcat['PartType1/Masses'] * 1e10 / h

        if len(masses) < 32:
            dens.append(0)
            slope.append(0)
            continue
        
        all_r = [1.3,1.5,1.7]
        tmp_dens = []
        for r in all_r:
            points = fibonacci_sphere(100, r)
            tmp_dens.append(calc_density(coords, masses, points, 32))
        
        dens.append(np.log10(np.average(tmp_dens)))
        # dens.append(tmp_dens)

        d5 = np.log10(tmp_dens[0])
        d10 = np.log10(tmp_dens[2])
        slope.append((d5 - d10) / (np.log10(all_r[0]) - np.log10(all_r[2])))

    return dens, slope

In [ ]:
def set_ticks(ax):
    ax.tick_params('both', which='minor', length=4, direction='in', bottom=True, top=True, left=True, right=True)
    ax.tick_params('both', which='major', length=8, direction='in', bottom=True, top=True, left=True, right=True)

    for axis in ['top','bottom','left','right']:
        ax.spines[axis].set_linewidth(2)

    return

def set_plot_params(nrows=1, ncols=1, figsize=None, **kwargs):

    plt.rc('font',**{'family':'STIXGeneral'})
    plt.rc('mathtext', **{'fontset':'stix'})
    plt.rc('text', usetex=False)

    plt.rc('font', size=12)          # controls default text sizes
    plt.rc('axes', titlesize=16)     # fontsize of the axes title
    plt.rc('axes', labelsize=24)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=18)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=18)    # fontsize of the tick labels
    plt.rc('legend', fontsize=16)

    plt.rc('lines', linewidth=3)

    if figsize is None:
        fig, ax = plt.subplots(nrows,ncols, figsize=(ncols*3 + (ncols)*3,nrows*5+(nrows-1)*3))
    else:
        fig, ax = plt.subplots(nrows,ncols, figsize=figsize, **kwargs)

    if type(ax)==type(np.zeros(1)):
        for a in ax.ravel():
            set_ticks(a)
    else:
        set_ticks(ax)

    return fig, ax

In [ ]:
def get_box_cut(part_pos, center, length):

    if type(length) == type([]):
        xlen = length[0]
        ylen = length[1]
        zlen = length[2]
    else:
        xlen = ylen = zlen = length

    xpr = center[0] + xlen
    xmr = center[0] - xlen
    ypr = center[1] + ylen
    ymr = center[1] - ylen
    zpr = center[2] + zlen
    zmr = center[2] - zlen

    is_x_in_box = (part_pos[:, 0] < xpr) & (part_pos[:, 0] > xmr)
    is_y_in_box = (part_pos[:, 1] < ypr) & (part_pos[:, 1] > ymr)
    is_z_in_box = (part_pos[:, 2] < zpr) & (part_pos[:, 2] > zmr)

    return is_x_in_box & is_y_in_box & is_z_in_box

def get_rotate_data(coords, velocities, masses, phi=0, theta=0, edge_on=False, face_on=False, get_pt=False, r_max=5, r_min=0):

    x = coords[:,0]
    y = coords[:,1]
    z = coords[:,2]
    vx = velocities[:,0]
    vy = velocities[:,1]
    vz = velocities[:,2]

    if edge_on or face_on:

        r2 = x*x+y*y+z*z
        scut = (r2<r_max*r_max) & (r2>r_min*r_min)

        lz = np.sum(masses[scut] * (x[scut]*vy[scut] - y[scut]*vx[scut]))
        lx = np.sum(masses[scut] * (y[scut]*vz[scut] - z[scut]*vy[scut]))
        ly = np.sum(masses[scut] * (z[scut]*vx[scut] - x[scut]*vz[scut]))

        phi = np.arctan2(ly, lx)
        theta = np.arctan2(np.sqrt(lx*lx + ly*ly), lz)

        if edge_on:
            theta += np.pi/2

    if get_pt:
        return phi, theta

    x_ = -z  * np.sin(theta) + (x * np.cos(phi) + y *np.sin(phi)) * np.cos(theta)
    y_ = -x  * np.sin(phi)   + y  * np.cos(phi)
    z_ =  z  * np.cos(theta) + (x * np.cos(phi) + y *np.sin(phi)) * np.sin(theta)
    vx_ = -vz  * np.sin(theta) + (vx * np.cos(phi) + vy *np.sin(phi)) * np.cos(theta)
    vy_ = -vx  * np.sin(phi)   + vy  * np.cos(phi)
    vz_ =  vz  * np.cos(theta) + (vx * np.cos(phi) + vy *np.sin(phi)) * np.sin(theta)

    coords[:,0] = x_
    coords[:,1] = y_
    coords[:,2] = z_

    velocities[:,0] = vx_
    velocities[:,1] = vy_
    velocities[:,2] = vz_

    return coords, velocities

In [ ]:
import sys, os
sys.path.append(TORREYLABTOOLS + '/Python/')
os.environ['PYDIR'] = TORREYLABTOOLS + '/Python'

In [ ]:
import util.calc_hsml as calc_hsml
import visualization.contour_makepic as cmakepic
from copy import copy

In [ ]:
def mag_to_lum(V, mag_sun):

    lums = np.zeros(V.shape[0])
    for i in range(V.shape[0]):
        mag = V[i]
        lums[i] = np.power(10, (mag - mag_sun)/2.5*-1)
    return lums

def lum_to_mag(lum, mag_sun):

    mags = np.zeros(lum.shape[0])
    for i in range(lum.shape[0]):
        l = lum[i]
        mags[i] = np.log10(l) * -2.5 + mag_sun

    return mags

In [ ]:
def get_r(pcat, cat, fov=20, face_on=False, edge_on=False):

    coords = copy(pcat['PartType4/Coordinates'])/h
    vels = copy(pcat['PartType4/Velocities'])
    masses = pcat['PartType4/Masses']*1e10/h
    phots = pcat['PartType4/GFM_StellarPhotometrics']
    gal_pos = cat['SubhaloPos']/h
    gal_vel = cat['SubhaloVel']
    gal_lum = cat['SubhaloStellarPhotometrics']
    gal_rad = cat['SubhaloHalfmassRadType'][4]/h

    fov = gal_rad*2
    
    lums = mag_to_lum(phots[:, 2], 4.8) #convert to luminosity 4.65=r - Willmer 2018

    avg_lum = np.average(lums)
    min_mag = lum_to_mag(np.array([np.sum(lums)+avg_lum/2]), 4.8)
    max_mag = lum_to_mag(np.array([np.sum(lums)-avg_lum/2]), 4.8)

    box_cut = get_box_cut(coords, gal_pos, fov)

    if np.sum(box_cut) < 2:
        return 0.6,0.6,0.6,0.6

    coords -= gal_pos
    vels -= gal_vel

    shalo_lum = mag_to_lum(np.array([gal_lum[2]]), 4.8)[0]
    lums = lums[box_cut]

    hsml = calc_hsml.get_particle_hsml(coords[box_cut,0], coords[box_cut,1], coords[box_cut,2], DesNgb=32)
    massmap,image = cmakepic.simple_makepic(coords[box_cut,0], coords[box_cut,1],
            weights=lums, hsml=hsml, xrange=[-fov,fov],yrange=[-fov,fov], pixels=512)

    im_r = 0
    im_lum = 0
    x = np.arange(512)
    y = np.arange(512)
    while im_lum <= np.sum(massmap)/2:
        cut = (x[np.newaxis,:]-256)**2 + (y[:,np.newaxis]-256)**2 < im_r**2
        im_lum = np.sum(massmap[cut])
        im_r += 1

    all_r = np.arange(256)
    all_l = []
    for r in all_r:
        cut = (x[np.newaxis,:]-256)**2 + (y[:,np.newaxis]-256)**2 < r**2
        all_l.append(np.sum(massmap[cut])/np.sum(massmap))

    all_l = np.array(all_l)
    lum_r = all_r[all_l<0.5][-1]
    err_r = (all_r[all_l<0.6][-1] - all_r[all_l<0.4][-1])/2

    #half-lum rad from star particles
    r=0.01
    star_lum = 0
    while star_lum <= shalo_lum/2:
        star_r2 = np.sum(np.square(coords[box_cut,:2]), axis=1)
        scut = star_r2 < r*r
        star_lum = np.sum(lums[scut])
        r += .01

    return lum_r/(512/(fov*2)), err_r/(512/(fov*2)), min_mag[0], max_mag[0] # im_r/(512/20) # r

In [ ]:
def get_r2(pcat, cat, fov=20, face_on=False, edge_on=False):

    coords = copy(pcat['PartType4/Coordinates'])/h
    vels = copy(pcat['PartType4/Velocities'])
    masses = pcat['PartType4/Masses']*1e10/h
    phots = pcat['PartType4/GFM_StellarPhotometrics']
    gal_pos = cat['SubhaloPos']/h
    gal_vel = cat['SubhaloVel']
    gal_lum = cat['SubhaloStellarPhotometrics']
    gal_rad = cat['SubhaloHalfmassRadType'][4]/h

    fov = gal_rad*3
    
    lums = mag_to_lum(phots[:, 5], 4.65) #convert to luminosity 4.65=r - Willmer 2018 V=4.8

    avg_lum = np.average(lums)
    min_mag = lum_to_mag(np.array([np.sum(lums)+avg_lum/2]), 4.8)
    max_mag = lum_to_mag(np.array([np.sum(lums)-avg_lum/2]), 4.8)

    box_cut = get_box_cut(coords, gal_pos, fov)

    if np.sum(box_cut) < 2:
        return -1

    coords -= gal_pos
    vels -= gal_vel

    shalo_lum = mag_to_lum(np.array([gal_lum[2]]), 4.8)[0]
    lums = lums[box_cut]

    hsml = calc_hsml.get_particle_hsml(coords[box_cut,0], coords[box_cut,1], coords[box_cut,2], DesNgb=32)
    massmap,image = cmakepic.simple_makepic(coords[box_cut,0], coords[box_cut,1],
            weights=lums, hsml=hsml, xrange=[-fov,fov],yrange=[-fov,fov], pixels=512)

    r, props = calculate_circularized_half_light_radius(massmap, fov/512)
    return r
    
    im_r = 0
    im_lum = 0
    x = np.arange(512)
    y = np.arange(512)
    while im_lum <= np.sum(massmap)/2:
        cut = (x[np.newaxis,:]-256)**2 + (y[:,np.newaxis]-256)**2 < im_r**2
        im_lum = np.sum(massmap[cut])
        im_r += 1

    all_r = np.arange(256)
    all_l = []
    for r in all_r:
        cut = (x[np.newaxis,:]-256)**2 + (y[:,np.newaxis]-256)**2 < r**2
        all_l.append(np.sum(massmap[cut])/np.sum(massmap))

    all_l = np.array(all_l)
    lum_r = all_r[all_l<0.5][-1]
    err_r = (all_r[all_l<0.6][-1] - all_r[all_l<0.4][-1])/2

    fig, ax = plt.subplots()

    #cut = (x[np.newaxis,:]-256)**2 + (y[:,np.newaxis]-256)**2 < im_r**2
    #massmap[~cut] = 0
    ax.imshow(massmap, norm=LogNorm())
    
    ax.plot(all_r, all_l)
    ax.plot([0,256],[.4,.4],'k--')
    ax.plot([0,256],[.5,.5],'k--')   
    ax.plot([0,256],[.5,.5],'k--')

    plt.show()

    #half-lum rad from star particles
    r=0.01
    star_lum = 0
    while star_lum <= shalo_lum/2:
        star_r2 = np.sum(np.square(coords[box_cut,:2]), axis=1)
        scut = star_r2 < r*r
        star_lum = np.sum(lums[scut])
        r += .01

    return lum_r/(512/(fov*2)), err_r/(512/(fov*2)), min_mag[0], max_mag[0] # im_r/(512/20) # r

In [ ]:
from astropy.io import fits
from astropy.modeling.models import Gaussian2D
from astropy.stats import sigma_clipped_stats
from photutils.aperture import EllipticalAperture
from photutils.segmentation import SourceCatalog, detect_sources
from scipy.optimize import brentq
import matplotlib

def calculate_circularized_half_light_radius(image_data, scale_kpc_per_pixel=1.0):
    """
    Calculates the circularized half-light radius for the brightest galaxy in a 2D image.

    This function follows a common methodology:
    1.  Estimate and subtract the background sky level.
    2.  Detect sources and identify the brightest one (assumed to be the target galaxy).
    3.  Measure the galaxy's shape (ellipticity, position angle) and total luminosity.
    4.  Iteratively find the semi-major axis of the ellipse that contains half the total light.
    5.  Convert this semi-major axis to a circularized radius.

    Args:
        image_data (np.ndarray): The 2D numpy array representing the galaxy image.
        scale_kpc_per_pixel (float): The physical scale of the image in kiloparsecs per pixel.

    Returns:
        float: The circularized half-light radius in kiloparsecs.
        dict: A dictionary containing intermediate properties for plotting or inspection.
    """
    # 1. Estimate and subtract the background
    mean, median, std = sigma_clipped_stats(image_data, sigma=1.0)
    background_level = median
    image_sub = image_data - background_level
    
    # 2. Detect sources to find the galaxy
    # The detection threshold is set to 1.5 sigma above the background noise
    threshold = 1.0 * std
    segment_map = detect_sources(image_sub, threshold, npixels=10)
    
    if segment_map is None:
        print("No sources detected.")
        return None, None
        
    # Create a source catalog and get the properties of the brightest source
    cat = SourceCatalog(image_sub, segment_map)
    brightest_source = cat[np.argmax([s.max_value for s in cat])]
    
    # 3. Get galaxy properties from the catalog
    center_x, center_y = brightest_source.xcentroid, brightest_source.ycentroid
    position_angle = brightest_source.orientation.to('rad').value
    ellipticity = brightest_source.ellipticity
    print(ellipticity)
    # Axial ratio q = b/a = 1 - ellipticity
    axial_ratio = 1 - ellipticity
    semi_major_axis_sigma = brightest_source.semimajor_sigma.value

    # 4. Calculate total luminosity within a large aperture (e.g., 5 * sigma)
    total_aperture_a = 5 * semi_major_axis_sigma

    if total_aperture_a <= 0.1:
        print(f"Source is too small or compact to measure half-light radius. (semimajor_sigma: {semi_major_axis_sigma:.2f})")
        return None, None
    
    total_aperture_b = total_aperture_a * axial_ratio

    print(total_aperture_a, total_aperture_b, position_angle, axial_ratio, semi_major_axis_sigma)

    pos_x = float(brightest_source.xcentroid)
    pos_y = float(brightest_source.ycentroid)
    aperture_a = float(total_aperture_a)
    aperture_b = float(total_aperture_b)
    angle = float(position_angle)

    large_aperture = EllipticalAperture(
        (pos_x, pos_y), aperture_a, aperture_b, theta=angle
    )
    
    # large_aperture = EllipticalAperture(
    #     (brightest_source.xcentroid, brightest_source.ycentroid), total_aperture_a, total_aperture_b, theta=position_angle
    # )
    total_flux = large_aperture.do_photometry(image_sub)[0][0]
    half_light_flux = total_flux / 2.0

    # 5. Iteratively find the semi-major axis that encloses half the light
    # We define a function whose root is the half-light radius.
    def flux_difference(a):
        b = a * axial_ratio
        a = float(a)
        b = float(b)
        position_angle = float(angle)
        aperture = EllipticalAperture((center_x, center_y), a, b, theta=angle)
        flux_in_aperture = aperture.do_photometry(image_sub)[0][0]
        return flux_in_aperture - half_light_flux

    # Use a robust root-finding algorithm (Brent's method) to find the semi-major axis
    # We search for the root in the interval [0.1 pixels, total_aperture_a]
    #try:
    #    r_eff_semimajor_pixels = brentq(flux_difference, 0.1, total_aperture_a)
    #except ValueError:
    #    print("Could not find half-light radius. The flux might not be monotonically increasing.")
    #    return None, None

    r=3
    flux_diff = -1e10
    while flux_diff < 0:
        flux_diff = flux_difference(r)
        r += .1

    r_eff_semimajor_pixels = r
        
    # 6. Circularize the radius and convert to physical units (kpc)
    # R_circ = R_eff,sma * sqrt(b/a)
    r_eff_circ_pixels = r_eff_semimajor_pixels * np.sqrt(axial_ratio)
    r_eff_circ_kpc = r_eff_circ_pixels * scale_kpc_per_pixel

    fig, ax = plt.subplots()
    ax.imshow(image_data) #, norm=LogNorm())
    debug_ellipse = matplotlib.patches.Ellipse(
                (brightest_source.xcentroid, brightest_source.ycentroid),
                width=total_aperture_a,
                height=total_aperture_b,
                angle=position_angle*180/np.pi,
                edgecolor='cyan',
                facecolor='none',
                lw=2,
                label='Total Flux Aperture'
            )
    ax.add_patch(debug_ellipse)
    half_light_ellipse = matplotlib.patches.Ellipse(
            (brightest_source.xcentroid, brightest_source.ycentroid),
            width=r_eff_semimajor_pixels,
            height=r_eff_semimajor_pixels*axial_ratio,
            angle=position_angle*180/np.pi,
            edgecolor='red',
            facecolor='none',
            lw=2,
            label='Half-Light Aperture'
        )
    ax.add_patch(half_light_ellipse)
    for s in cat:
        ax.plot(s.xcentroid, s.ycentroid, 'k.')
    plt.show()
    
    properties = {
        'center_x': center_x,
        'center_y': center_y,
        'position_angle_rad': position_angle,
        'axial_ratio': axial_ratio,
        'r_eff_semimajor_pixels': r_eff_semimajor_pixels,
        'r_eff_circ_pixels': r_eff_circ_pixels,
        'total_flux': total_flux,
        'background_level': background_level,
        'image_subtracted': image_sub
    }
    
    return r_eff_circ_kpc, properties


In [ ]:
sat_mass = []
sat_smass = []
sat_half_smass = []
sat_half_lum = []
sat_pos = []
sat_vel = []
sat_a10 = []
sat_a50 = []
sat_a90 = []
sat_a100 = []
sat_dens = []
sat_slope = []
sat_Mk = []
sat_params = []
sat_boxes = []
gal_smass = []
gal_mass = []
nsubs = []
plt.figure()
for box in boxes[:20]:

    if box % 50 == 0:
        print(box)
    
    path = f'{group_path}/box_{box}/fof_subhalo_tab_{snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{snap:03}.hdf5'
        
    cat = load_group_data(path, keys)     #read DREAMS data
    mw_idx = get_MW_idx(cat)        #find target MW
    sat_cat = get_sats(cat, mw_idx, rlim=400) #, rlim=1000, pos_cut=True) #get satellite information
    pcat = load_particle_data(spath, part_keys, [1,4])

    sat_idx = get_sat_idx(cat, mw_idx)

    if len(sat_cat['SubhaloMass']) <= 1:
        continue

    all_r = []
    for sub_idx in sat_idx:
        part_cat, group_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx, sub_idx=sub_idx)
        smass = group_cat['SubhaloMassType'][4]*1e10/h
        if smass < 5e9:
            continue
        # r, err, mi, ma = get_r2(part_cat, group_cat)
        r = get_r2(part_cat, group_cat)
        # plt.plot(np.log10(smass), r, 'k.')
    continue
    #     all_r.append(r)
    # all_r = np.array(all_r)
    # print(all_r)
    # np.save(f"data/rads_circ/box_{box}", all_r)
    # continue
    all_r = np.load(f"data/rads_circ/box_{box}.npy")

    smass = sat_cat['SubhaloMassType'][1:,4] * 1e10 / h
    scut = smass > 1e6

    if np.sum(scut) == 0:
        print("!!!!", box)
        continue
    
    gpos = cat['GroupPos'][mw_idx]
    gvel = cat['GroupVel'][mw_idx]

    #a10, a50, a90, a100 = calc_sfh(pcat, cat, mw_idx, sat_idx)
    dens, slope = calc_dm(pcat, cat, mw_idx, sat_idx)
    dens = np.array(dens)
    slope = np.array(slope)

    pos = (sat_cat['SubhaloPos'][1:][scut] - gpos) / h / 1e3
    vel = (sat_cat['SubhaloVel'][1:][scut] - gvel) / h
    mass = sat_cat['SubhaloMass'][1:][scut] * 1e10 / h
    smass = sat_cat['SubhaloMassType'][1:,4][scut] * 1e10 / h

    gal_mass.append(cat['Group_M_Crit200'][mw_idx]*1e10/h)
    gal_smass.append(cat['GroupMassType'][mw_idx,4]*1e10/h)

    #sat_a10.append(a10)
    #sat_a50.append(a50)
    #sat_a90.append(a90)
    #sat_a100.append(a100)
    sat_dens.append(dens[scut])
    sat_slope.append(slope[scut])
    sat_half_lum.append(all_r)
    
    # sfr = sat_cat['SubhaloSFR']
    # sfr[sfr==0] = np.random.rand()*1e-5

    # gmass = sat_cat['SubhaloMassType'][:,0] * 1e10 / h
    # gmass[gmass==0] = np.random.rand()*1e5
    
    # sat_sfr.append(sfr[1:])
    # sat_gmass.append(gmass[1:])
    sat_pos.append(pos)
    sat_vel.append(vel)
    sat_mass.append(mass)
    sat_smass.append(smass)
    # sat_Mk.append(sat_cat['SubhaloStellarPhotometrics'][1:,3])
    # sat_half_smass.append(sat_cat['SubhaloHalfmassRadType'][1:,4] / h)
    sat_params.append(params[box])
    sat_boxes.append(box)
    nsubs.append(len(sat_cat['SubhaloMass'][1:][scut]))

In [ ]:
plt.figure()
for box in range(60):
    new =  np.load(f"data/rads_circ/box_{box}.npy")[0]
    old =  np.load(f"data/rads/box_{box}.npy")[0]
    plt.plot(old, new, 'k.')
plt.plot([0,5],[0,5],'r--')
plt.xlabel("Old")
plt.ylabel("New")
plt.show()

In [ ]:
nsubs = np.log10(np.array(nsubs))
om = np.array(sat_params)[:,0]
s8 = np.array(sat_params)[:,1]
sn1 = np.log10(np.array(sat_params)[:,2])
sn2 = np.log10(np.array(sat_params)[:,3])
agn1 = np.log10(np.array(sat_params)[:,4])
smass = np.log10(np.array(gal_smass)+1)
mass = np.log10(np.array(gal_mass)+1)

In [ ]:
data_cond = np.array([nsubs,smass, mass, om, s8, sn1, sn2, agn1]).T

In [ ]:
np.max([len(el) for el in sat_dens])

In [ ]:
data = np.zeros((len(sat_pos), 50, 11))
mask = np.zeros((len(sat_pos), 50), dtype=bool)

In [ ]:
for i in range(10):
    print(len(sat_smass[i]), len(sat_pos[i]))

In [ ]:
for i in range(len(sat_smass)):
    nsat = len(sat_smass[i])
    
    data[i, :nsat, :3] = sat_pos[i]
    data[i, :nsat, 3:6] = sat_vel[i]
    
    data[i, :nsat, 6] = np.log10(sat_mass[i])
    data[i, :nsat, 7] = np.log10(sat_smass[i]+1)
    # data[i, :nsat, 8] = sat_a10[i]
    # data[i, :nsat, 9] = sat_a50[i]
    # data[i, :nsat, 10] = sat_a90[i]
    # data[i, :nsat, 11] = sat_a100[i]
    data[i, :nsat, 8] = sat_dens[i]
    data[i, :nsat, 9] = sat_slope[i]
    data[i, :nsat, 10] = sat_half_lum[i]
    
    # data[i, :nsat, 8] = np.log10(sat_gmass[i])

    # data[i, :nsat, 8] = np.log10(sat_sfr[i])
    # data[i, :nsat, 10] = np.log10(sat_half_smass[i])
    # data[i, :nsat, 9] = sat_Mk[i]

    mask[i, :nsat] = [True]*nsat

In [ ]:
data_cond.shape, data.shape, mask.shape

In [ ]:
header='nsubs,smass,mass,om,s8,sn1,sn2,agn1'
np.savetxt("data/master_sat_cdm_circ_cond.csv", data_cond, delimiter=',', header=header, comments='')
np.savez('data/master_sat_cdm_circ', features=data, mask=mask)

In [ ]:
print(data[0,0])

In [ ]:
plt.figure()
plt.plot(data[mask,7], data[mask,9], 'k.')
plt.xlabel("Satellite Stellar Mass")
plt.ylabel("SFR")
plt.show()

In [ ]:
plt.figure()
plt.plot(data[mask,7], data[mask,8], 'k.')
plt.ylabel("Satellite Gas Mass")
plt.xlabel("Satellite Stellar Mass")
plt.show()

In [ ]:
def load_disk(gpath, spath):

    cat = load_group_data(gpath, keys)     #read DREAMS data
    pcat = load_particle_data(spath, part_keys, part_types=[4])
    mw_idx = get_MW_idx(cat)        #find target MW
    
    part_cat, group_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx)
    
    coords = part_cat['PartType4/Coordinates'] - group_cat['GroupPos']
    vels = part_cat['PartType4/Velocities'] - group_cat['GroupVel']
    masses = part_cat['PartType4/Masses'] * 1e10
    
    coords, vels = get_rotate_data(coords, vels, masses, face_on=True, edge_on=False)

    return coords, vels, masses

In [ ]:
def mag_to_lum(V, mag_sun):
    lums = np.power(10, -(V - mag_sun)/2.5)
    return lums

def lum_to_mag(lums, mag_sun):
    mags = -2.5*np.log10(lums)+mag_sun
    return mags

def calc_r_opt(pcat, gcat):

    pt = "PartType4/"

    coords = pcat[pt+'Coordinates']/h - gcat['GroupPos']/h
    vels = pcat[pt+'Velocities'] - gcat['GroupVel']
    mass = pcat[pt+'Masses']*1e10/h
    mag = pcat[pt+'GFM_StellarPhotometrics'][:,1]

    lums = mag_to_lum(mag, 5.31) #B=5.31 - Willmer 2018 

    coords, vels = get_rotate_data(coords, vels, mass, face_on=True)
    star_r2 = np.sum(np.square(coords[:,:2]), axis=1)

    Sigma = 0
    r = 0
    while Sigma < 25:
        if r >= 50:
            return r
        r += .1
        scut = (star_r2 > r*r) & (star_r2 < (r+1)*(r+1))

        annuli_lum = np.sum(lums[scut])
        Sigma_kpc = annuli_lum / (np.pi*((r+1)**2-r**2)*1e6)
        Sigma = 5.31  + 21.572 - 2.5*np.log10(Sigma_kpc)

    return r

def calc_m_opt(pcat, gcat, r_opt):

    mass_tot = dict()
    for i in [0,1,4]:
        pt = f"PartType{i}/"
        coords = pcat[pt+'Coordinates']/h - gcat['GroupPos']/h
        vels = pcat[pt+'Velocities'] - gcat['GroupVel']
        mass = pcat[pt+'Masses']*1e10/h

        coords, vels = get_rotate_data(coords, vels, mass, face_on=True)
        r2 = np.sum(np.square(coords[:,:2]), axis=1)

        mass_tot[i] = np.sum(mass[r2 < r_opt*r_opt])

    return mass_tot

In [ ]:
def calc_surface_dens_profile(pcat, gcat, min_r, max_r, r_step=1.05, part_types=[4], zheight=5):
    gal_pos = gcat['GroupPos']/h
    gal_vel = gcat['GroupVel']

    pos = pcat['PartType4/Coordinates']/h
    mass = pcat['PartType4/Masses']*1e10/h
    vel = pcat['PartType4/Velocities']

    pos -= gal_pos
    vel -= gal_vel
    
    pos, vel = get_rotate_data(pos, vel, mass, face_on=True)

    zcut = (pos[:,2] > -zheight) & (pos[:,2] < zheight)
    pos = pos[zcut]
    mass = mass[zcut]
    part_r2 = np.sum(np.square(pos), axis=1)

    dens_li = []
    all_r = []
    prev_r = 0
    r = min_r
    while(r < max_r):
        r2 = r*r
        pr2 = prev_r*prev_r
        scut = (part_r2<r2) & (part_r2 > pr2)
        in_s = pos[scut]
        in_s_mass = mass[scut]
        out_area = np.pi*r2
        in_area = np.pi*pr2
        in_mass = np.sum(in_s_mass)

        dens_li.append(in_mass / (out_area - in_area) / 1e6)
        all_r.append(r)

        prev_r = r
        r *= r_step

    return np.array(all_r), np.array(dens_li)

def lin(x, m, b):
    return m*x+b

def get_disk_frac(coords, vels, masses):

    Lz = coords[:,0]*masses*vels[:,1] - coords[:,1]*masses*vels[:,0]
    Lmax = np.sqrt(np.sum(np.square(vels), axis=1)) * masses * np.sqrt(np.sum(np.square(coords), axis=1))

    Lcut = Lmax != 0
    epsilon = Lz[Lcut] / Lmax[Lcut]

    disk_cut = epsilon > 0.7

    return np.sum(disk_cut) / len(disk_cut)

def get_disk_size(cat, pcat, mw_idx):
    part_cat, group_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx)

    r, d = calc_surface_dens_profile(part_cat, group_cat, 0.1, 30, part_types=[4], r_step=1.05)
    
    rcut = (r>3) & (d > 1)
    #if np.sum(rcut) < 2:
    #    continue

    x = r[rcut]
    y = np.log(d[rcut])
    
    #plt.figure()
    #plt.plot(r, np.log10(d), '.')
    #plt.plot(x, y, '.')
    
    guess = [-0.1, 2]
    popt, pcov = curve_fit(lin, x, y) #, guess)
    m, b = popt
    
    scale_len = -1/m
    #plt.plot(x, lin(x,m,b), 'k-')
    #plt.plot(x, x*guess[0]+guess[1], 'r-')
    #plt.show()

    return scale_len


def calc_tully_fisher(cat, pcat, mw_idx):
    part_cat, group_cat = get_galaxy_data(pcat, cat, fof_idx=mw_idx)
    
    coords = part_cat['PartType4/Coordinates'] - group_cat['GroupPos']
    vels = part_cat['PartType4/Velocities'] - group_cat['GroupVel']
    masses = part_cat['PartType4/Masses'] * 1e10
    
    r_opt = calc_r_opt(part_cat, group_cat)

    mass = calc_m_opt(part_cat, group_cat, r_opt)
    m_opt_all = np.sum([mass[pt] for pt in [0,1,4]])
    v = np.sqrt(G*m_opt_all / r_opt)

    return np.log10(v), np.sum([mass[pt] for pt in [0,4]])

In [ ]:
def calc_density(matter_coords, matter_mass, sample_coords, DesNgb):
    """
    This function takes particle data and calculates the density around a given set of points.
    The points are assumed to be for one radius and are averaged together before returned.
    This function will work for any particle type given to it.
    
    Inputs
      matter_coords - the coordinates for the simulation particles used to calculate the density
      matter_mass   - the mass of the simulation particles, must be the same shape as matter_coords
      sample_coords - the coordinates where the density is calculated
      DesNgb        - the number of simulation particles used to calculate the density (32 is standard)
      
    Returns
      density - the average density for the supplied sample coordinates
    """
    tree = KDTree(matter_coords)
    distance, idx = tree.query(sample_coords, DesNgb)
    hsml = distance[:,-1]
    mass_enclosed = np.sum(matter_mass[idx], axis=1)
    density =  mass_enclosed / (4 / 3 * np.pi * np.power(hsml,3))
    
    #density = np.average(density)
    return density

def make_image(coords, vels, masses, fov, pixels):

    cut = get_box_cut(coords, np.zeros(3), fov*1.1)
    
    coords = coords[cut]
    masses = masses[cut]

    points1 = np.linspace(-fov, fov, pixels)
    points2 = np.linspace(-fov/2, fov/2, 60)
    mesh_points = np.meshgrid(points1, points1, points2)
    mesh_points = np.array(mesh_points)
    
    xpoints = mesh_points[0].ravel('C')
    ypoints = mesh_points[1].ravel('C')
    zpoints = mesh_points[2].ravel('C')
    sample_points = np.array([xpoints, ypoints, zpoints]).T
    
    density_map = calc_density(coords, masses, sample_points, 16)
    
    massmap = density_map.reshape(pixels, pixels, 60, order='C')
    massmap = np.average(massmap, axis=2)

    return massmap

In [ ]:
def age_from_a(a, a0=1, H0=69.09, Om0=0.3, Ode0=0.7):
      OmegaLambda = Ode0
      Omega0      = Om0

      factor1 = 2.0 / (3.0 * np.sqrt(OmegaLambda));

      term1 = np.sqrt(OmegaLambda / Omega0) * a0**1.5
      term2 = np.sqrt(1 + OmegaLambda / Omega0 * a0**3.0)
      factor2 = np.log(term1 + term2);

      t0 = factor1 * factor2;

      term1 = np.sqrt(OmegaLambda / Omega0) * a**1.5
      term2 = np.sqrt(1 + OmegaLambda / Omega0 * a**3.0)
      factor2 = np.log(term1 + term2);

      t1 = factor1 * factor2;

      result = t0 - t1;

      hubble            = 3.2407789e-18
      hubbleparam       = H0/100
      sec_per_megayear  = 3.15576e13

      ft = result / (hubble * hubbleparam);     #/* now in seconds */
      ft /= sec_per_megayear * 1000;            #a  /* now in gigayears */

      return ft

def calc_stellar_age(cat, pcat, mw_idx, Om):

    ages = pcat['PartType4/GFM_StellarFormationTime']
    coords = pcat['PartType4/Coordinates']
    masses = pcat['PartType4/Masses']*1e10/h
    
    r = np.sqrt(np.sum(np.square(coords - cat['GroupPos'][mw_idx]),axis=1))/h
    rcut = r < 50

    star_cut = ages > 0
    star_ages = ages[rcut & star_cut]

    ages = age_from_a(star_ages, Om0=Om, Ode0=1-Om) 
    masses = masses[rcut & star_cut]

    return np.sum(ages*masses) / np.sum(masses)
    

In [ ]:
def get_ellipse(gpath, spath, pixels, fov):

    coords, vels, masses = load_disk(gpath, spath)
    
    for j in range(2):

        massmap = make_image(coords, vels, masses, fov, pixels)
        
        lg_massmap = np.log10(massmap)
        
        guess = EllipseGeometry(pixels//2, pixels//2, 30, 0.5, -.5)
        ellipse = Ellipse(lg_massmap, guess)
        isolist = ellipse.fit_image(minsma=5, step=[10,2][j], linear=True)

        if len(isolist)==0:
            return None, None
        
        s = []
        f = []
        for i in isolist:
            s.append(i.sma)
            f.append(i.tflux_e / i.npix_e)
        
        skip = 1
        s = np.array([np.average(s[i-skip:i+skip]) for i in range(skip,len(s)-skip)])
        f = np.array([np.average(f[i-skip:i+skip]) for i in range(skip,len(f)-skip)])

        if j == 0:
            # plt.plot(s, f/np.max(f), '-')
    
            idx = np.argmin(np.abs(f/np.max(f)-0.85))
            fov = isolist[idx].sma/pixels*fov*2
            continue

        # plt.plot((s[1:]+s[:-1])/2, f[:-1]-f[1:], '-')
        
        idx = np.argmax(f[:-1]-f[1:])
        iso = isolist.get_closest(s[idx])
        
    return iso.sma*fov/pixels*2, iso.eps

def calc_bar(gpath, spath):

    fov = 14
    pixels = 128

    bar_length, elip = get_ellipse(gpath, spath, pixels, fov)

    return bar_length, elip

In [ ]:
def calc_smass(pcat, cat):

    rad = cat['SubhaloHalfmassRadType'][4]
    r = np.sqrt(np.sum(np.square(pcat['PartType4/Coordinates']-cat['SubhaloPos']), axis=1))
    rcut = r < 2*rad

    return np.sum(pcat['PartType4/Masses'][rcut]*1e10/h)


def get_reff(part_cat, group_cat):

    r, d = calc_surface_dens_profile(part_cat, group_cat, 0.1, 30, part_types=[4], r_step=1.05)
    
    rcut = (r<4) & (r>0.4) & (d > 1)

    x = r[rcut]
    y = np.log(d[rcut])
    
    guess = [-0.3, 8]
    popt, pcov = curve_fit(lin, x, y) #, guess)
    m, b = popt
    
    scale_len = -1/m

    # plt.figure()
    # plt.plot(r, np.log(d), '.')
    # plt.plot(x, y, '.')
    # plt.plot(x, lin(x,m,b), 'k-')
    # plt.plot(x, x*guess[0]+guess[1], 'r-')
    # plt.show()

    return scale_len

def calc_vel_disp(pcat, cat, reff):

    r = np.sqrt(np.sum(np.square(pcat['PartType4/Coordinates']-cat['SubhaloPos'][0]), axis=1))
    rcut = r < 2.0*reff

    vels = pcat['PartType4/Velocities'][rcut,0] - cat['SubhaloVel'][0][0]

    return np.log10(np.std(vels))

def calc_OH(pcat, cat):

    rad = cat['SubhaloHalfmassRadType'][0,4]
    r = np.sqrt(np.sum(np.square(pcat['PartType0/Coordinates']-cat['SubhaloPos'][0]), axis=1))
    rcut = r < 2*rad

    XO = np.sum(pcat['PartType0/GFM_Metals'][rcut,4]) #/ np.sum(cat['PartType0/GFM_Metals'][sub_idx,2:])
    XH = np.sum(pcat['PartType0/GFM_Metals'][rcut,0])
    #Z = np.sum(cat['PartType0/GFM_Metals'][sub_idx,2:])

    OH_frac = np.log10(XO/XH/16)+12

    return OH_frac

def load_sublink_cat(path, keys):
    """
    Read Sublink Merger Trees from the DREAMS simulations
    
    Inputs
      path - the absolute or relative path to the hdf5 file you want to read from
      keys - the data that you want to read from the simulation 
             see https://www.tng-project.org/data/docs/specifications/ for a list of available data)
      
    Returns
      cat - a dictionary that contains all of the information for the specified keys
    """
    cat = dict()
    with h5py.File(path, 'r') as ofile:
        for key in keys:
            cat[key] = np.array(ofile[key])
    return cat

def get_idx(box,mw_idx,oh_snap):

    tree_keys = ['SubhaloID','SubhaloGrNr','SnapNum','FirstSubhaloInFOFGroupID','MainLeafProgenitorID','FirstProgenitorID','SubfindID']
    tpath = f'{group_path}/box_{box}/tree_extended.hdf5'
    tree_cat = load_sublink_cat(tpath, tree_keys)

    if len(tree_cat['SubhaloGrNr']) != len(tree_cat['SnapNum']):
        return dict()
    
    nnodes = len(tree_cat['SubhaloID'])

    #initialize the search by finding the target halo in the tree
    target_cut = np.in1d(tree_cat['SubhaloGrNr'], mw_idx) & np.in1d(tree_cat['SnapNum'], 90)
    target  = tree_cat['FirstSubhaloInFOFGroupID'][target_cut][0]
    target_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == target][0]

    next_ID = tree_cat['FirstProgenitorID'][target_idx]
    next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]
    
    #first get the mass of the central for every snapshot
    idx_over_time = {tree_cat['SnapNum'][target_idx]:tree_cat['SubfindID'][target_idx]}
    while next_ID != -1:
        this_snap = tree_cat['SnapNum'][next_idx]
        this_mass = tree_cat['SubfindID'][next_idx]
        idx_over_time[this_snap] = this_mass
        
        next_ID = tree_cat['FirstProgenitorID'][next_idx]
        if next_ID == -1:
            break
        next_idx = np.arange(nnodes)[tree_cat['SubhaloID'] == next_ID][0]

    return idx_over_time

def calc_OH2(idx_over_time,oh_snap):
    
    gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{oh_snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{oh_snap:03}.hdf5'
    
    cat = load_group_data(gpath, keys)     #read DREAMS data
    pcat = load_particle_data(spath, part_keys, part_types=[0,4])
    galaxy_pcat, galaxy_cat = get_galaxy_data(pcat, cat, -1, idx_over_time[oh_snap])
    
    rad = galaxy_cat['SubhaloHalfmassRadType'][4]
    r = np.sqrt(np.sum(np.square(galaxy_pcat['PartType0/Coordinates']-galaxy_cat['SubhaloPos']), axis=1))
    rcut = r < 2*rad

    XO = np.sum(galaxy_pcat['PartType0/GFM_Metals'][rcut,4]) #/ np.sum(cat['PartType0/GFM_Metals'][sub_idx,2:])
    XH = np.sum(galaxy_pcat['PartType0/GFM_Metals'][rcut,0])
    #Z = np.sum(cat['PartType0/GFM_Metals'][sub_idx,2:])

    OH_frac = np.log10(XO/XH/16)+12

    rstar = np.sqrt(np.sum(np.square(galaxy_pcat['PartType4/Coordinates']-galaxy_cat['SubhaloPos']), axis=1))
    rcut = rstar < 2*rad
    smass = np.sum(galaxy_pcat['PartType4/Masses'])*1e10/h

    return OH_frac, smass

def calc_bh_mass(pcat, cat):

    masses = pcat['PartType5/Masses']*1e10/h
    nbh = len(masses)

    if nbh == 1:
        return masses[0]
    if nbh == 0:
        return 0
    
    r = np.sqrt(np.sum(np.square(pcat['PartType5/Coordinates']-cat['SubhaloPos'][0]), axis=1))
    idx = np.argmin(r)

    return masses[idx]

In [ ]:
keys = ['GroupMassType', 'GroupLenType', 'GroupPos', 'SubhaloGrNr', 'SubhaloPos', 'SubhaloMassType', 'SubhaloLenType', 'GroupFirstSub', 'GroupNsubs']
#keys += ['SubhaloMass', 'SubhaloPos', 'SubhaloVel', 'GroupPos', 'GroupVel', 'SubhaloHalfmassRadType', 'GroupMass', 'SubhaloGasMetalFractions', 'SubhaloVelDisp', 'SubhaloGasMetallicity', 'Group_M_Crit200','SubhaloVmax']
keys += ['GroupVel','SubhaloVel','SubhaloVmax','SubhaloHalfmassRadType','Group_M_Crit200','SubhaloSFR','Group_M_TopHat200']
part_keys = ['Coordinates', 'Masses', 'Velocities', 'GFM_Metals']

In [ ]:
disk_size = []
disk_frac = []
tully_fisher = []
# nsubs = []
sim_params = []
smass = []
smass1 = []
mass = []
gmass = []
smass_rad = []
stellar_age = []
avg_Z_gas = []
avg_Z_star = []
sfr = []
vel_disp = []
bh_mass = []
bar_elip = []
bar_length = []
central_boxes = []
tully_fisher_mass = []
tully_fisher_vel = []
Mk = []
reff = []
OH_frac = []
SFR = []
OH_smass = []
for box in boxes:
    gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{snap:03}.hdf5'

    if box % 50 == 0:
        print(box)
    
    if not (os.path.exists(gpath) and os.path.exists(spath)):
        continue

    if box == 698:
        continue
    
    cat = load_group_data(gpath, keys)     #read DREAMS data
    # pcat = load_particle_data(spath, part_keys, part_types=[0,1,4])
    mw_idx = get_MW_idx(cat)        #find target MW
    sub_idx = cat['GroupFirstSub'][mw_idx]
    # galaxy_pcat, galaxy_cat = get_galaxy_data(pcat, cat, mw_idx, -1)
    # sat_cat = get_sats(cat, mw_idx) #get satellite information

    #length, elip = calc_bar(gpath, spath)
    #if length is None:
    #    continue

    #this_smass = calc_smass(galaxy_pcat, galaxy_cat)
    # if this_smass < 1e5:
    #     print(box, "###")
    #     continue

    # this_reff = get_reff(galaxy_pcat, galaxy_cat)

    # if this_reff < 0:
    #     print(box, "!!!")
    #     continue

    # this_vel_disp = calc_vel_disp(galaxy_pcat, galaxy_cat, this_reff)
    
    # oh_snap = [67,73,81,90][np.random.randint(4)]
    # print(box, oh_snap)
    # idx_over_time = get_idx(box,mw_idx,oh_snap)
    # print(idx_over_time)
    # if oh_snap not in idx_over_time:
    #     continue
    # print(box, "!!!")
    # OH, oh_smass = calc_OH2(idx_over_time,oh_snap)
    # sfr = cat['SubhaloSFR'][sub_idx]

    # vel_disp.append(this_vel_disp) #cat['SubhaloVelDisp'][sub_idx])
    # OH_frac.append(OH)
    # OH_smass.append(oh_smass)
    # SFR.append(sfr)
    
    # reff.append(this_reff)

    smass.append(cat['SubhaloMassType'][sub_idx,4]*1e10/h)
    #smass.append(this_smass)
    # mass.append(cat['Group_M_Crit200'][mw_idx]*1e10/h)
    mass.append(cat['Group_M_TopHat200'][mw_idx]*1e10/h)
    sim_params.append(params[box])
    central_boxes.append(box)

    #vmax = cat['SubhaloVmax'][sub_idx]
    #bmass = cat['SubhaloMassType'][sub_idx,0] + cat['SubhaloMassType'][sub_idx,4]
    #bmass *= 1e10 / h
    #tully_fisher_mass.append(bmass)
    #tully_fisher_vel.append(vmax)
    
    # vel_disp.append(this_vel_disp)
    # bh_mass.append(cat['SubhaloMassType'][sub_idx,5]*1e10/h)
    # OH_frac.append(OH)
    continue


    
    #bar_elip.append(elip)
    #bar_length.append(length)

    Om = params[box][0]
    #mean_age = calc_stellar_age(cat, pcat, mw_idx, Om)
    #stellar_age.append(mean_age)
    
    avg_Z_gas.append(cat['SubhaloGasMetalFractions'][sub_idx,4] / cat['SubhaloGasMetalFractions'][sub_idx,0])
    # avg_Z_star.append(cat['SubhaloStarMetalFractions'][sub_idx,4] / cat['SubhaloStarMetalFractions'][sub_idx,0])
    # sfr.append(cat['SubhaloSFR'][sub_idx])
    # nsubs.append(len(sat_cat['SubhaloMass'])-1)
    mass.append(cat['GroupMass'][mw_idx]*1e10/h)
    # Mk.append(cat['SubhaloStellarPhotometrics'][sub_idx,3])
    sim_params.append(params[box])
    central_boxes.append(box)
    continue

    
    smass_rad.append(cat['SubhaloHalfmassRadType'][sub_idx,4] / h)
    
    
    size = get_disk_size(cat, pcat, mw_idx)
    coords, vels, masses = load_disk(gpath, spath)
    frac = get_disk_frac(coords, vels, masses)

    # fig, ax = plt.subplots(1,2,figsize=(6,3))
    # print(size, frac)
    # massmap = make_image(coords, vels, masses, 30, 128)
    # massmap = np.log10(massmap)
    # ax[0].imshow(massmap)
    # coords, vels = get_rotate_data(coords, vels, masses, face_on=False, edge_on=True)
    # massmap = make_image(coords, vels, masses, 30, 128)
    # massmap = np.log10(massmap)
    # ax[1].imshow(massmap)
    # x = np.linspace(-1*size/15*128,size/15*128, 20)
    # ax[0].plot(x+64, np.sqrt(size*size/(30*30)*(128*128) - x*x)+64, 'r-')
    # ax[0].plot(x+64, -1*np.sqrt(size*size/(30*30)*(128*128) - x*x)+64, 'r-')
    # plt.show()
    # plt.close(fig)
    
    tullyf, tullyf_mass = calc_tully_fisher(cat, pcat, mw_idx)
    tully_fisher.append(tullyf)
    tully_fisher_mass.append(tullyf_mass)
    
    disk_size.append(size)
    disk_frac.append(frac)
    
    gmass.append(cat['GroupMassType'][mw_idx,0]*1e10/h)
    

In [ ]:
plt.plot(smass, OH_smass, 'k.')
plt.xscale("log")
plt.yscale("log")

In [ ]:
plt.hist(OH_frac)

In [ ]:
SFR = np.array(SFR)
smass = np.array(smass)
OH_frac = np.array(OH_frac)
cut = SFR < 0.05
print(np.sum(cut) / len(cut))
plt.figure()
plt.plot(smass, OH_frac, '.')
plt.plot(smass[~cut], OH_frac[~cut], '.')
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(np.log10(mass), np.log10(smass), 'k.')
plt.xlim(11.5,12.5)
plt.ylabel("M$_*$ [M$_\odot$]")
plt.ylabel("M$_\mathrm{halo}$ [M$_\odot$]")
plt.show()

In [ ]:
x, mean, high, low = get_obs('massZ', 9, 11.7, 10)
fig = plt.figure()
plt.plot(np.log10(smass), OH_frac, 'k.')
plt.xlim(9,11.5)
plt.plot(x, mean, 'r--')
plt.plot(x, high, 'r-')
plt.plot(x, low, 'r-')
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("log(O/H)+12")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(smass, reff, 'k.')
plt.xscale("log")
plt.ylim(0,3)
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(vel_disp, bh_mass, 'k.')
# plt.xscale("log")
plt.ylim(5,9)
plt.show()

In [ ]:
fig = plt.figure()
bl = np.array(bar_length)
be = np.array(bar_elip)
plt.plot(bl, be, 'k.')
plt.xlim(0,20)
plt.ylim(0,1)
plt.ylabel("Ellipticity")
plt.xlabel("Bar Length [kpc]")
plt.show()

In [ ]:
fig = plt.figure()
bl = np.array(bar_length)
be = np.array(bar_elip)
bar_cut = (bl < 10) & (be > 0.5)
bins = np.logspace(10,11.3,10)
avg, be, bn = binned_statistic(smass, bar_cut.astype(int), 'mean', bins=bins)
plt.plot((bins[1:]+bins[:-1])/2, avg, 'k.')
plt.xlim(1e10,2e11)
plt.ylim(0,1)
plt.plot([1.7,2.6],[6,10], 'r-')
plt.xscale("log")
#plt.yscale("log")
plt.xlabel("log M$_\mathrm{*}$ [M$_\odot$]")
plt.ylabel("Bar Fraction")
plt.show()

In [ ]:
fig = plt.figure()
bl = np.array(bar_length)
be = np.array(bar_elip)
bar_cut = (bl < 10) & (be > 0.5)
plt.plot(np.array(smass)[bar_cut], bl[bar_cut], 'k.')
plt.xlim(1e9,2e11)
plt.ylim(0.5,20)
# plt.plot([1.7,2.6],[6,10], 'r-')
plt.xscale("log")
plt.yscale("log")
plt.xlabel("log M$_\mathrm{*}$ [M$_\odot$]")
plt.ylabel("Bar Length")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(np.log10(vel_disp), np.log10(bh_mass), 'k.')
plt.xlim(1.5,2.5)
plt.ylim(5,11)
plt.plot([1.7,2.6],[6,10], 'r-')
#plt.xscale("log")
#plt.yscale("log")
plt.ylabel("log M$_\mathrm{BH}$ [M$_\odot$]")
plt.xlabel("log $\sigma_\mathrm{x}$ [km s$^{-1}$]")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(smass, np.array(sfr)/np.array(smass), 'k.')
plt.ylim(2e-12,1e-9)
plt.xlim(1e9,3e11)
# plt.plot([1e10,3e10,2e11],[-.3,0,0.1], 'r-')
plt.xscale("log")
plt.yscale("log")
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("sSFR [yr$^{-1}$]")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(smass, np.array(Mk), 'k.')
# plt.ylim(2e-12,1e-9)
plt.xlim(1e9,3e11)
# plt.plot([1e10,3e10,2e11],[-.3,0,0.1], 'r-')
plt.xscale("log")
# plt.yscale("log")
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("Mk")
plt.show()

In [ ]:
def func(x):
    return 8.793 - (.28/1.2)*np.log(1+np.power(x/10.02,-1.2))

def get_obs(name, mi, ma, npoints=5):

    mean = np.loadtxt(f'data/obsdata/{name}_mean.csv', delimiter=',') #, skiprows=1)
    high = np.loadtxt(f'data/obsdata/{name}_high.csv', delimiter=',') #, skiprows=1)
    low = np.loadtxt(f'data/obsdata/{name}_low.csv', delimiter=',') #, skiprows=1)
    
    x = np.linspace(mi, ma, npoints)
    me = np.interp(x, mean[:,0], mean[:,1])
    hi = np.interp(x, high[:,0], high[:,1])
    lo = np.interp(x, low[:,0], low[:,1])
    
    return x, me, hi, lo

In [ ]:
x, mean, high, low = get_obs('massZ', 9, 11.7, 10)
# mean -= 8.7
# low -= 8.7
# high -= 8.7
# low = mean + np.sqrt(mean - low)
# high = mean - np.sqrt(high - mean)

In [ ]:
fig = plt.figure()
plt.plot(np.log10(smass), np.log10(np.array(avg_Z_gas))+12, 'k.')
# plt.plot(np.log10(smass), np.log(np.array(avg_Z_star))+12, 'k.')
# plt.ylim(-1.1,1.0)
plt.xlim(9,11.5)
plt.plot(x, mean, 'r--')
plt.plot(x, high, 'r-')
plt.plot(x, low, 'r-')
# plt.plot([1e9,1e10,2e11],[8.5,8.71,8.79], 'r-')
# plt.plot([1e9,1e10,2e11],[8.5+.07,8.71+.07,8.79+.07], 'r--')
# plt.plot([1e9,1e10,2e11],[8.5-.07,8.71-.07,8.79-.07], 'r--')
# plt.xscale("log")
# plt.ylim(7,9)
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("log(O/H)+12")
plt.show()

In [ ]:
fO = []
XH = []
Z = []
smass = []
sim_params = []
central_boxes = []
nsubs = []
for box in boxes[1:]:
    gpath = f'{group_path}/box_{box}/fof_subhalo_tab_{snap:03}.hdf5'
    spath = f'{snap_path}/box_{box}/snap_{snap:03}.hdf5'
    if not (os.path.exists(gpath) and os.path.exists(spath)):
        continue
    
    cat = load_group_data(gpath, keys)     #read DREAMS data
    mw_idx = get_MW_idx(cat)        #find target MW
    sat_cat = get_sats(cat, mw_idx) #get satellite information

    sub_idx = cat['GroupFirstSub'][mw_idx]

    if cat['SubhaloGasMetalFractions'][sub_idx,0] == 0:
        print(box)
        continue
    
    smass.append(cat['SubhaloMassType'][sub_idx,4]*1e10/h)
    fO.append(cat['SubhaloGasMetalFractions'][sub_idx,4] / np.sum(cat['SubhaloGasMetalFractions'][sub_idx,2:]))
    XH.append(cat['SubhaloGasMetalFractions'][sub_idx,0])
    Z.append(cat['SubhaloGasMetallicity'][sub_idx])
    sim_params.append(params[box])
    central_boxes.append(box)
    nsubs.append(len(sat_cat['SubhaloMass'])-1)

In [ ]:
fO = np.array(fO)
XH = np.array(XH)
Z = np.array(Z)
OH_frac = np.log10(Z*fO/XH/16)+12

In [ ]:
x, mean, high, low = get_obs('massZ', 9, 11.7, 10)
fig = plt.figure()
plt.plot(np.log10(smass), np.log10(Z*fO/XH/16)+12, 'k.')
plt.xlim(9,11.5)
plt.plot(x, mean, 'r--')
plt.plot(x, high, 'r-')
plt.plot(x, low, 'r-')
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("log(O/H)+12")
plt.show()

In [ ]:
plt.figure()
bins = np.logspace(-3,0,50)
plt.hist(oxy, bins, label='Oxygen')
plt.hist(hyd, bins, label='Hydrogen')
plt.xscale("log")
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(smass, stellar_age, 'k.')
plt.ylim(0,14)
plt.xlim(1e9,5e11)
plt.xscale("log")
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("age [Gyr]")
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(smass, smass_rad, 'k.')
plt.xscale("log")
plt.yscale("log")
plt.ylim(0.5,20)
plt.xlim(1e9,5e11)
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("R$_{*}^{1/2}$ [kpc]")
plt.show()

In [ ]:
good_size = [2.5,6.9,4.9,5.2,3.7,6.9,5.2,3.2,4.2,5.6,9.7,6.0,1.7,2.5,2.4,5.1,3.3,6.3,4.0,2.9,4.2,5.7,3.3,3.7,4.8,4.9,6.8,4.6,4.6,]
good_frac = [.59,.57,.51,.58,.45,.51,.48,.48,.34,.44,.53,.43,.42,.23,.64,.27,.70,.56,.32,.47,.34,.64,.61,.61,.60,.41,.67,.38,.41,]
maybe_size = [5.0,3.5,4.4,2.3,2.8,2.5,3.8,3.0,]
maybe_frac = [.24,.15,.38,.28,.20,.36,.48,.22,]
bad_size = [2.9,2.3,2.3,2.8,3.3,3.8,2.7,2.8,1.9,2.7,12.9,2.5,2.0,2.9,2.4,2.2,3.3,2.5,2.1,2.4,3.2,3.7,2.9,2.0,2.8,3.1,4.0,]
bad_frac = [.14,.07,.13,.38,.27,.30,.17,.42,.36,.28,0.05,.27,.11,.22,.27,.06,.20,.11,.21,.42,.14,.28,.25,.16,.17,.21,.23,]
fig = plt.figure()
plt.plot(good_size, good_frac, 'g+')
plt.plot(maybe_size, maybe_frac, 'yo')
plt.plot(bad_size, bad_frac, 'rx')

s = np.linspace(0,15,16)
plt.plot(s[s<=3],[0.45]*4,'k-')
plt.plot(s[(s>=3) & (s<=8)],-.025*s[(s>=3) & (s<=8)]+0.525,'k-')
plt.plot(s[s>=8],[0.325]*8,'k-')
plt.xlabel("Disk Size [kpc]")
plt.ylabel("Disk Mass Fraction")
plt.show()
fig.savefig("plots/disk_vs_bulge.pdf", bbox_inches='tight')

In [ ]:
plt.figure()

plt.plot(smass, np.log10(disk_size), '.')
plt.show()

In [ ]:
plt.figure()

plt.plot(np.log10(tully_fisher_mass), tully_fisher, '.')
plt.plot(10,2.05, 'kx')
plt.show()

In [ ]:
vel_disp = np.log10(np.array(vel_disp))
bh_mass = np.log10(np.array(bh_mass))

In [ ]:
central_boxes = np.array(central_boxes)
sat_boxes = np.array(sat_boxes)
nan_cut = ~np.isnan(OH_frac) & (np.array(bh_mass)>=0)
central_cut = np.in1d(central_boxes, sat_boxes) #& nan_cut
sat_cut = np.in1d(sat_boxes, central_boxes) #[nan_cut])
print(np.sum(central_cut), np.sum(sat_cut))

In [ ]:
central_cut = np.ones(len(central_boxes),dtype=bool)

In [ ]:
data.shape, mask.shape

In [ ]:
# nsubs = np.log10(np.array(nsubs))[sat_cut]
om = np.array(sim_params)[central_cut,0]
s8 = np.array(sim_params)[central_cut,1]
sn1 = np.log10(np.array(sim_params)[central_cut,2])
sn2 = np.log10(np.array(sim_params)[central_cut,3])
agn1 = np.log10(np.array(sim_params)[central_cut,4])
smass = np.log10(np.array(smass))[central_cut]
mass = np.log10(np.array(mass))[central_cut]
# gmass = np.log10(np.array(gmass))
# bar_length = np.array(bar_length)
# # bar_elip = np.array(bar_elip)
# vel_disp = np.log10(np.array(vel_disp))
# bh_mass = np.log10(np.array(bh_mass))
# # sfr = np.array(sfr)[central_cut]
# # Mk = np.array(Mk)[central_cut]
# # avg_Z = np.log10(np.array(avg_Z)/.0014)
# # stellar_age = np.array(stellar_age)
# # disk_size = np.array(disk_size)
# # disk_frac = np.array(disk_frac)
# tully_fisher_vel = np.array(tully_fisher_vel)
# tully_fisher_mass = np.log10(np.array(tully_fisher_mass))
# data_cond = np.array([nsubs, smass, mass, disk_size, disk_frac, tully_fisher,tully_fisher_mass, bar_length, bar_elip, vel_disp, bh_mass, sfr, avg_Z, stellar_age, om, s8, sn1, sn2, agn1]).T
# data_cond = np.array([nsubs, smass, mass, sfr, Mk, om, s8, sn1, sn2, agn1]).T
data_cond = np.array([nsubs, smass, mass, om, s8, sn1, sn2, agn1]).T

In [ ]:
np.save('../../ceph/nehod_ethan/example/data/om', om)
np.save('../../ceph/nehod_ethan/example/data/s8', s8)
np.save('../../ceph/nehod_ethan/example/data/sn1', sn1)
np.save('../../ceph/nehod_ethan/example/data/sn2', sn2)
np.save('../../ceph/nehod_ethan/example/data/agn', agn1)
np.save('../../ceph/nehod_ethan/example/data/smass', smass)
np.save('../../ceph/nehod_ethan/example/data/mass', mass)

In [ ]:
nsubs[central_cut].shape

In [ ]:
np.sum(np.isnan(np.array(OH_frac)[central_cut]))

In [ ]:
mass

In [ ]:
# nsubs = np.log10(np.array(nsubs))
om = np.array(sim_params)[:,0]
s8 = np.array(sim_params)[:,1]
sn1 = np.log10(np.array(sim_params)[:,2])
sn2 = np.log10(np.array(sim_params)[:,3])
agn1 = np.log10(np.array(sim_params)[:,4])
smass = np.log10(np.array(smass)+1)
mass = np.log10(np.array(mass)+1)
# OH_frac = np.array(OH_frac)
# bh_mass = np.log10(np.array(bh_mass)+1)
# vel_disp = np.array(vel_disp)
# tully_fisher_vel = np.log10(np.array(tully_fisher_vel))
# tully_fisher_mass = np.log10(np.array(tully_fisher_mass))
# OH_smass = np.log10(np.array(OH_smass)+1)
# data_cond = np.array([smass, mass, bh_mass, vel_disp, OH_frac, OH_smass, om, s8, sn1, sn2, agn1]).T
data_cond = np.array([smass, mass, om, s8, sn1, sn2, agn1]).T

In [ ]:
data_cond = np.array([smass, mass, bh_mass, vel_disp, OH_frac, OH_smass, om, s8, sn1, sn2, agn1]).T

In [ ]:
data_cond = np.loadtxt("data/master_sat_cdm_cond.csv", delimiter=',', skiprows=1)
data_cond.shape

In [ ]:
plt.hist(OH_frac)

In [ ]:
OH_frac = OH_frac[::2]

In [ ]:
plt.hist(OH_frac)

In [ ]:
[np.sum(np.isnan(data_cond[:,i])) for i in range(11)]

In [ ]:
data_cond[np.isnan(data_cond)]

In [ ]:
cut = ~np.isnan(OH_frac)
print(np.sum(cut))
data_cond = data_cond[cut]

In [ ]:
data_cond.shape

In [ ]:
np.sum(np.isnan(data_cond))

In [ ]:
print(data_cond.shape)

In [ ]:
x, mean, high, low = get_obs('massZ', 9, 11.7, 10)
fig = plt.figure()
plt.plot(smass, OH_frac, 'k.')
plt.xlim(9,11.5)
plt.plot(x, mean, 'r--')
plt.plot(x, high, 'r-')
plt.plot(x, low, 'r-')
plt.xlabel("M$_*$ [M$_\odot$]")
plt.ylabel("log(O/H)+12")
plt.show()

In [ ]:
print(data[sat_cut].shape, data_cond.shape)

In [ ]:
# np.savez('data/master_cdm', features=data[sat_cut], mask=mask[sat_cut])
data_cond = np.array([smass, mass, bh_mass, vel_disp, OH_frac, OH_smass, om, s8, sn1, sn2, agn1]).T

In [ ]:
header = 'stellar_mass,total_mass,bh_mass,vel_disp,OH_frac,OH_smass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
# np.savetxt("data/master_cdm_highz_cond.csv", data_cond, delimiter=',', header=header, comments='')

In [ ]:
header = 'log_num_subhalos,stellar_mass,total_mass,disk_size,disk_frac,tully_fisher,tully_fisher_mass,bar_length,bar_elip,vel_disp,bh_mass,sfr,avg_Z,stellar_age,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
# np.savetxt("data/master_cdm_cond.csv", data_cond[central_cut], delimiter=',', header=header, comments='')

In [ ]:
header = 'log_num_subhalos,stellar_mass,total_mass,sfr,Mk,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
# np.savetxt("data/Mk_cdm_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.savez('data/Mk_cdm', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
header = 'log_num_subhalos,stellar_mass,total_mass,bh_mass,vel_disp,OH_frac,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
# np.savetxt("data/master_cdm_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.savez('data/master_cdm', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
header = 'stellar_mass,total_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
# np.savetxt("data/master_cdm_new_weight_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.savez('data/master_cdm', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
header = 'stellar_mass,total_mass,bh_mass,vel_disp,OH_frac,tf_vel,tf_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
np.savetxt("data/master_cdm_tf_cond.csv", data_cond, delimiter=',', header=header, comments='')
# np.savez('data/master_cdm_tf', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
header = 'log_num_subhalos,stellar_mass,total_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
np.savetxt("data/master_sat_cdm_cond.csv", data_cond, delimiter=',', header=header, comments='')
np.savez('data/master_sat_cdm', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
header = 'log_num_subhalos,stellar_mass,total_mass,omega_m,sigma_8,log_sn1,log_sn2,log_agn1'
np.savetxt("data/master_sat_nocut_cdm_cond.csv", data_cond, delimiter=',', header=header, comments='')
np.savez('data/master_sat_nocut_cdm', features=data[sat_cut], mask=mask[sat_cut])

In [ ]:
np.average(data_cond,axis=0)

In [ ]:
ar = np.load('data/master_sat_cdm.npz')

In [ ]:
ar['features'][0,0]

In [ ]:
plt.figure()
bins = np.linspace(6,10,20)
bin_center = (bins[1:] + bins[:-1]) / 2
smass_func = []
for i in range(len(ar['features'])):
    sm = ar['features'][i,:,7]
    avg, be, bn = binned_statistic(sm, sm, 'count', bins=bins)
    smass_func.append(avg/(bins[1]-bins[0]))
avg = np.average(smass_func, axis=0)
std = np.sqrt(np.average(np.square(smass_func - avg), axis=0))
plt.plot(bin_center, avg, 'k-') 
plt.fill_between(bin_center, avg+std, avg/((avg+std)/avg), alpha=0.3) 
plt.yscale("log")

In [ ]:
plt.figure()
rad_func = []
bins = np.linspace(0,300,50)
bin_center = (bins[1:]+bins[:-1])/2
for i in range(len(ar['features'])):
    r = np.sqrt(np.sum(np.square(ar['features'][i,:,:3]),axis=1))*1e3
    sat_r = r[r > 0]
    if len(sat_r) == 0:
        continue
    idx1 = np.argsort(sat_r)
    rad = np.interp(bin_center, sat_r[idx1], np.arange(len(sat_r)))
    if np.max(rad) == 0:
        continue
    rad_func.append(rad/np.max(rad))
avg = np.mean(rad_func, axis=0)
std = np.std(rad_func, axis=0)
plt.plot(bin_center, avg, 'k-') 
std_high = avg + std
std_high[std_high > 1] = 1
plt.fill_between(bin_center, std_high, avg/((avg+std)/avg), alpha=0.3) 

In [ ]:
mass = ar['features'][:,:,6]
dens = ar['features'][:,:,8]
slope = ar['features'][:,:,9]

sat_mass = mass[~np.isnan(mass) & (dens > 4)]
sat_slope = slope[~np.isnan(slope) & (dens > 4)]

bins = np.linspace(8.5,11,30)

avg, be, bn = binned_statistic(sat_mass, sat_slope, 'mean', bins=bins) #lambda x:np.quantile(x,0.5), bins=bins)
std, be, bn = binned_statistic(sat_mass, sat_slope, 'std', bins=bins) # lambda x:np.quantile(x,0.84), bins=bins)
# std_low, be, bn = binned_statistic(sat_mass, sat_slope, lambda x:np.quantile(x,0.16), bins=bins)
std_high = std + avg
std_low = avg - std

bin_center = (bins[1:] + bins[:-1]) / 2
plt.plot(bin_center, avg, f'k-') 
plt.fill_between(bin_center, std_low, std_high, alpha=0.4) 

In [ ]:
ar = np.load('data/Mk_cdm.npz')

In [ ]:
ar['features'][0,0]

In [ ]:
plt.figure()
smass_func = []
bins = np.linspace(6,10,20)
bin_center = (bins[1:] + bins[:-1]) / 2
for i in range(len(ar['features'])):
    sm = ar['features'][i,:,7]
    avg, be, bn = binned_statistic(sm, sm, 'count', bins=bins)
    smass_func.append(avg/(bins[1]-bins[0]))
avg = np.average(smass_func, axis=0)
std = np.sqrt(np.average(np.square(smass_func - avg), axis=0))
plt.plot(bin_center, avg, 'k-') 
plt.fill_between(bin_center, avg+std, avg/((avg+std)/avg), alpha=0.3) 
plt.yscale("log")

In [ ]:
plt.figure()
rad_func = []
bins = np.linspace(0,300,50)
bin_center = (bins[1:]+bins[:-1])/2
for i in range(len(ar['features'])):
    r = np.sqrt(np.sum(np.square(ar['features'][i,:,:3]),axis=1))*1e3
    sat_r = r[r > 0]
    if len(sat_r) == 0:
        continue
    idx1 = np.argsort(sat_r)
    rad = np.interp(bin_center, sat_r[idx1], np.arange(len(sat_r)))
    if np.max(rad) == 0:
        continue
    rad_func.append(rad/np.max(rad))
avg = np.mean(rad_func, axis=0)
std = np.std(rad_func, axis=0)
plt.plot(bin_center, avg, 'k-') 
std_high = avg + std
std_high[std_high > 1] = 1
plt.fill_between(bin_center, std_high, avg/((avg+std)/avg), alpha=0.3) 